# Optimizing prompts automatically — DSPy + MLflow

*Part of the `gen_ai/` track (advanced). Prerequisites: `c_genai_evaluation`, `e_prompt_registry`. The traditional-ML analog is `ml/b_hyperparameter_tuning`.*

## The problem: hand-tuning prompts doesn't scale

In `e_prompt_registry` you wrote prompt **v1**, then **v2**, and used the `c_` judge to pick the winner. That's *manual* prompt engineering — you guessed the edits. **DSPy** does the search for you: you specify a **task** and a **metric**, and an **optimizer** automatically improves the prompt — its instructions and its few-shot examples — against that metric. It's **hyperparameter tuning, but for prompts** (`ml/b_hyperparameter_tuning` did it for model knobs).

And because MLflow has a first-class DSPy integration, **`mlflow.dspy.autolog()` records the whole optimization** — every compile step, the evaluations, and the optimized program saved as an artifact — so you can see *how* it improved and reproduce it.

| Term | One-line meaning |
|---|---|
| **signature** | A typed task spec — input/output fields — instead of a raw prompt string. |
| **module** | A strategy that runs a signature (e.g. `ChainOfThought`). |
| **optimizer** (teleprompter) | The search that improves the prompt against a metric (e.g. `BootstrapFewShot`). |
| **compile** | Running the optimizer to produce an improved program. |

## Prerequisites

- A **tracking server on 5001**.
- **DSPy**: `uv add dspy` (already added).
- **A capable, fast model.** Optimization makes *many* model calls and DSPy parses structured output, so — exactly like `c_`'s judge — we use a hosted model (**Azure**), credentials from `.env`. A local reasoning model (qwen3) is slow here and its `<think>` output fights DSPy's parser; the Ollama config is shown commented for the patient.

**Diverges from upstream tutorial:** MLflow's DSPy tutorial uses OpenAI; we route DSPy's LM to **Azure** through LiteLLM (`azure/<deployment>`), reusing the repo's `.env`.

In [ ]:
import os, dspy, mlflow
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-dspy-optimization")
mlflow.dspy.autolog(log_compiles=True, log_evals=True)   # one line: autolog the optimization

lm = dspy.LM(
    f"azure/{os.environ.get('AZURE_OPENAI_LIGHT_MODEL', 'gpt-5.4-nano')}",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_base=os.environ["AZURE_OPENAI_BASE_URL"].removesuffix("/openai").rstrip("/"),
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)
# Local alternative (slower; qwen3's thinking can confuse DSPy's parser):
#   lm = dspy.LM("ollama_chat/qwen3:8b", api_base="http://localhost:11434")
dspy.configure(lm=lm)

## Define the task as a program, not a prompt

A **`Signature`** declares the task by its fields; **`ChainOfThought`** is a module that answers it with step-by-step reasoning. You never write the prompt string — DSPy builds (and later *optimizes*) it from the signature.

In [ ]:
class QA(dspy.Signature):
    """Answer the question with a short factual answer (a few words)."""
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()

program = dspy.ChainOfThought(QA)
print(program(question="What is the capital of Australia?").answer)

## A tiny dataset and a metric

The optimizer needs examples to learn from (`trainset`) and a held-out set to score (`devset`), plus a **metric** — the same idea as a validation metric in `ml/`. Ours: is the expected answer contained in the prediction?

In [ ]:
pairs = [("France","Paris"), ("Japan","Tokyo"), ("Italy","Rome"), ("Spain","Madrid"),
         ("Germany","Berlin"), ("Canada","Ottawa"), ("Australia","Canberra"),
         ("Brazil","Brasilia"), ("Egypt","Cairo"), ("Turkey","Ankara")]
data = [dspy.Example(question=f"What is the capital of {c}?", answer=a).with_inputs("question")
        for c, a in pairs]
trainset, devset = data[:6], data[6:]

def metric(example, pred, trace=None):
    return example.answer.lower() in (pred.answer or "").lower()

## Score the un-optimized program

`dspy.Evaluate` runs the program over the devset and reports the metric — the **baseline** to beat.

In [ ]:
from dspy.evaluate import Evaluate

evaluator = Evaluate(devset=devset, metric=metric, num_threads=1, display_progress=False)
print("baseline:", evaluator(program))

## Optimize — and watch MLflow record it

`BootstrapFewShot` runs the program on the trainset, keeps the traces that pass the metric, and folds them back in as **few-shot demonstrations**. Because `mlflow.dspy.autolog()` is on, MLflow opens a run and logs the optimizer's parameters, the compile, and the optimized program as a JSON artifact.

In [ ]:
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(metric=metric, max_bootstrapped_demos=2, max_labeled_demos=2)
optimized = optimizer.compile(program, trainset=trainset)
print("bootstrapped demos:", len(optimized.predictors()[0].demos))

## Score the optimized program

Same devset, same metric — now with the learned few-shot demos baked into the prompt.

In [ ]:
print("optimized:", evaluator(optimized))

# the demos DSPy *learned* and now prepends to every prompt:
for d in optimized.predictors()[0].demos[:2]:
    print("-", d.get("question"), "->", d.get("answer"))

## What MLflow captured

Open <http://127.0.0.1:5001> → the `genai-dspy-optimization` experiment. The autologged run holds the optimizer's parameters, the per-step evaluations, traces from the compile, and the **optimized program saved as an artifact** — so you can see *how* the prompt improved, not just *that* it did, and load the exact optimized program later.

On this easy task the score may already be near-perfect — the lesson is the **mechanism**: DSPy searched the prompt for you and MLflow recorded the search. On harder tasks (reasoning, formatting, domain conventions) the gains are larger and the MLflow record is what makes them reproducible.

## Where this sits — and the road not taken

- **vs `e_prompt_registry`:** there you hand-versioned prompts; here DSPy *learns* one. Natural combo: optimize with DSPy, then `register_prompt` the result so `f_genai_app_serving` can serve it by alias.
- **vs `ml/b_hyperparameter_tuning`:** same MLOps move — search a space against a metric, with MLflow tracking every trial — applied to prompts instead of model hyperparameters.
- **The alternative we considered — [AdalFlow](https://github.com/SylphAI-Inc/AdalFlow):** a *"PyTorch for LLMs"* library that optimizes via **textual gradients** (an auto-diff graph over the pipeline). Conceptually elegant for an ML audience and Ollama-native, but its MLflow integration isn't first-class, so for an *MLflow* tutorial DSPy's native `autolog` wins. Worth a look if you like the autograd framing.

This is the advanced cap of the `gen_ai/` track.